In [ ]:
import sys
from pathlib import Path
 
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
 

import json
import time
import pandas as pd
 
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from jiwer import cer

from src.data import (
    build_vocab_from_df,
    make_collate_fn,
    load_lang_data,
    MultilingualASRDataset,
)
from src.model_baseline import HuBERT_CTC
from src.model_lora import HuBERT_CTC_LoRA
from src.model_houlsby import HuBERT_CTC_Houlsby
from src.train_utils import (
    set_seed,
    train_epoch,
    evaluate,
    save_checkpoint,
    count_parameters,
)
from src.eval_utils import (
    run_inference,
    save_results_json,
    save_predictions_csv,
)

In [ ]:
TRAIN_LANGS = ["swa", "kat", "kir", "tam"]
EVAL_LANG = "swa"
 
SHARED_CONFIG = {
    "model_name": "utter-project/mHuBERT-147-base-3rd-iter",
    "data_size": "10min",
    "batch_size": 8,
    "grad_accumulation": 4,
    "learning_rate": 1e-4,
    "weight_decay": 1e-6,
    "target_iterations": 1500,
    "eval_every": 5,
}
 
LORA_CONFIG = {
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.1,
    "target_modules": ["q_proj", "v_proj"],
}
 
HOULSBY_CONFIG = {
    "adapter_bottleneck": 32,
    "adapter_dropout": 0.1,
}

In [ ]:
DATA_ROOT = PROJECT_ROOT / "data" / "commonvoice"
RESULTS_DIR = PROJECT_ROOT / "results" / "multilingual"
CHECKPOINTS_DIR = PROJECT_ROOT / "checkpoints" / "multilingual"
 
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
all_train_dfs = []
eval_dev_df = None
eval_test_df = None
 
for lang in TRAIN_LANGS:
    print(f"\n── Loading {lang} ──")
    df_tr, df_dv, df_te = load_lang_data(lang, DATA_ROOT, SHARED_CONFIG["data_size"])
    all_train_dfs.append(df_tr)
 
    if lang == EVAL_LANG:
        eval_dev_df = df_dv
        eval_test_df = df_te
 
df_train_multi = pd.concat(all_train_dfs, ignore_index=True)
print(f"\n── Combined training: {len(df_train_multi)} samples from {len(TRAIN_LANGS)} languages ──")

In [ ]:
vocab = build_vocab_from_df(df_train_multi)
idx_to_char = {v: k for k, v in vocab.items()}
print(f"Vocab size (multilingual): {len(vocab)}")

In [ ]:
train_dataset = MultilingualASRDataset(df_train_multi)
dev_dataset = MultilingualASRDataset(eval_dev_df)
test_dataset = MultilingualASRDataset(eval_test_df)
 
collate = make_collate_fn(vocab)
 
train_loader = DataLoader(
    train_dataset,
    batch_size=SHARED_CONFIG["batch_size"],
    shuffle=True,
    collate_fn=collate,
    num_workers=0,
    pin_memory=True,
)
 
dev_loader = DataLoader(
    dev_dataset,
    batch_size=SHARED_CONFIG["batch_size"],
    shuffle=False,
    collate_fn=collate,
    num_workers=0,
    pin_memory=True,
)
 
test_loader = DataLoader(
    test_dataset,
    batch_size=SHARED_CONFIG["batch_size"],
    shuffle=False,
    collate_fn=collate,
    num_workers=0,
)

In [ ]:
num_train_samples = len(train_dataset)
effective_batch = SHARED_CONFIG["batch_size"] * SHARED_CONFIG["grad_accumulation"]
iterations_per_epoch = num_train_samples / effective_batch
num_epochs = int((SHARED_CONFIG["target_iterations"] / iterations_per_epoch) + 0.9999)
print(f"num_epochs = {num_epochs} "
      f"(~{SHARED_CONFIG['target_iterations']} iters, "
      f"{num_train_samples} samples, eff. batch {effective_batch})")

In [ ]:
def run_experiment(model, experiment_name, extra_config=None):
    """Full training + eval on EVAL_LANG test set. Returns results dict."""
 
    config = {
        **SHARED_CONFIG,
        "experiment_name": experiment_name,
        "train_langs": TRAIN_LANGS,
        "eval_lang": EVAL_LANG,
        "num_epochs": num_epochs,
        "vocab_size": len(vocab),
    }
    if extra_config:
        config.update(extra_config)
 
    model = model.to(device)
    count_parameters(model)
 
    criterion = nn.CTCLoss(blank=vocab["<blank>"], zero_infinity=True)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["learning_rate"],
        weight_decay=config["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=10,
    )
 
    best_dev_loss = float("inf")
    history = {"train_loss": [], "dev_loss": [], "lr": [], "epoch": []}
    best_ckpt_path = CHECKPOINTS_DIR / f"{experiment_name}_best.pt"
    start_time = time.time()
    total_iterations = 0
 
    for epoch in range(num_epochs):
        train_loss = train_epoch(
            model, train_loader, criterion, optimizer,
            device, config["grad_accumulation"],
        )
        total_iterations += len(train_loader) // config["grad_accumulation"]
 
        if (epoch + 1) % config["eval_every"] == 0 or epoch == 0 or epoch == num_epochs - 1:
            dev_loss = evaluate(model, dev_loader, criterion, device)
            scheduler.step(dev_loss)
            print(f"  [{experiment_name}] Epoch {epoch+1}/{num_epochs} "
                  f"train={train_loss:.4f}  dev={dev_loss:.4f}")
        else:
            dev_loss = history["dev_loss"][-1] if history["dev_loss"] else float("inf")
 
        history["train_loss"].append(train_loss)
        history["dev_loss"].append(dev_loss)
        history["lr"].append(optimizer.param_groups[0]["lr"])
        history["epoch"].append(epoch + 1)
 
        if dev_loss < best_dev_loss:
            best_dev_loss = dev_loss
            save_checkpoint(
                model=model, optimizer=optimizer, epoch=epoch,
                path=best_ckpt_path, dev_loss=dev_loss,
                config=config, vocab=vocab,
            )
 
    # Save history
    with open(RESULTS_DIR / f"{experiment_name}_history.json", "w") as f:
        json.dump(history, f, indent=2, ensure_ascii=False)
 
    # Eval on swa test set
    checkpoint = torch.load(best_ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
 
    predictions, references = run_inference(
        model=model, loader=test_loader, device=device,
        idx_to_char=idx_to_char, vocab=vocab,
    )
 
    char_error_rate = cer(references, predictions)
    elapsed = (time.time() - start_time) / 60
 
    results = {
        "experiment_name": experiment_name,
        "model_name": config["model_name"],
        "train_langs": TRAIN_LANGS,
        "eval_lang": EVAL_LANG,
        "data_size": config["data_size"],
        "num_train_samples": num_train_samples,
        "num_epochs": num_epochs,
        "target_iterations": config["target_iterations"],
        "actual_iterations": total_iterations,
        "best_dev_loss": best_dev_loss,
        "cer": char_error_rate,
        "training_time_minutes": elapsed,
    }
 
    save_results_json(results, RESULTS_DIR / f"{experiment_name}_results.json")
    save_predictions_csv(references, predictions,
                         RESULTS_DIR / f"{experiment_name}_predictions.csv")
 
    print(f"\n  >> {experiment_name}  CER={char_error_rate:.4f}  ({elapsed:.1f} min)\n")
    return results

In [ ]:
langs_tag = "_".join(TRAIN_LANGS)
 
model_baseline = HuBERT_CTC(
    vocab_size=len(vocab),
    model_name=SHARED_CONFIG["model_name"],
)
results_baseline = run_experiment(
    model_baseline,
    experiment_name=f"multi_{langs_tag}_baseline_{SHARED_CONFIG['data_size']}",
)

In [ ]:
model_lora = HuBERT_CTC_LoRA(
    vocab_size=len(vocab),
    model_name=SHARED_CONFIG["model_name"],
    lora_r=LORA_CONFIG["lora_r"],
    lora_alpha=LORA_CONFIG["lora_alpha"],
    lora_dropout=LORA_CONFIG["lora_dropout"],
    target_modules=LORA_CONFIG["target_modules"],
)
results_lora = run_experiment(
    model_lora,
    experiment_name=f"multi_{langs_tag}_lora_r{LORA_CONFIG['lora_r']}_{SHARED_CONFIG['data_size']}",
    extra_config=LORA_CONFIG,
)

In [ ]:
model_houlsby = HuBERT_CTC_Houlsby(
    vocab_size=len(vocab),
    model_name=SHARED_CONFIG["model_name"],
    adapter_bottleneck=HOULSBY_CONFIG["adapter_bottleneck"],
    adapter_dropout=HOULSBY_CONFIG["adapter_dropout"],
)
results_houlsby = run_experiment(
    model_houlsby,
    experiment_name=f"multi_{langs_tag}_houlsby_b{HOULSBY_CONFIG['adapter_bottleneck']}_{SHARED_CONFIG['data_size']}",
    extra_config=HOULSBY_CONFIG,
)

In [ ]:
summary = pd.DataFrame([results_baseline, results_lora, results_houlsby])
summary = summary[["experiment_name", "cer", "best_dev_loss",
                    "training_time_minutes", "num_train_samples"]]
print("\n" + "=" * 80)
print(f"MULTILINGUAL FINETUNING — Eval on {EVAL_LANG}")
print("=" * 80)
print(summary.to_string(index=False))
print("=" * 80)
 
summary.to_csv(RESULTS_DIR / f"summary_multi_{langs_tag}.csv", index=False)

In [ ]:
mono_results_dir = PROJECT_ROOT / "results"
mono_files = {
    "baseline_mono": mono_results_dir / "baseline" / f"mhubert_base_swa_{SHARED_CONFIG['data_size']}_results.json",
    "lora_mono": mono_results_dir / "lora" / f"mhubert_lora_swa_{SHARED_CONFIG['data_size']}_r8_results.json",
    "houlsby_mono": mono_results_dir / "houlsby" / f"mhubert_houlsby_swa_{SHARED_CONFIG['data_size']}_b32_results.json",
}

rows = []
for label, path in mono_files.items():
    if path.exists():
        with open(path) as f:
            r = json.load(f)
        rows.append({"method": label, "cer": r["cer"], "setting": "monolingual"})
    else:
        print(f"Not found: {path}")
 
for r in [results_baseline, results_lora, results_houlsby]:
    name = r["experiment_name"]
    method = "baseline" if "baseline" in name else "lora" if "lora" in name else "houlsby"
    rows.append({"method": f"{method}_multi", "cer": r["cer"], "setting": "multilingual"})
 
if rows:
    comparison = pd.DataFrame(rows)
    print(f"\n── Mono vs Multi CER on {EVAL_LANG} ──")
    print(comparison.to_string(index=False))
    comparison.to_csv(RESULTS_DIR / f"mono_vs_multi_{langs_tag}.csv", index=False)
